# Tata Steel Maintenance Wizard - Qwen3-8B QLoRA Fine-Tuning Notebook

This notebook fine-tunes **Qwen/Qwen3-8B** with LoRA adapters for the Maintenance Wizard backend.

It does **not** train on raw CSV rows directly. It first builds grounded instruction examples from:

- synthetic steel demo sensor/history/failure/spares/SOP data
- public AI4I benchmark converted to common schema
- dynamic asset memory and remembered safety rules
- decision-intelligence outputs: risk, RUL, procurement risk, evidence confidence, missing evidence

Recommended Kaggle accelerator: T4 x2, P100, A100, or better. If Qwen3-8B OOMs, reduce `MAX_LENGTH`, `LORA_R`, or use Qwen3-4B for testing.


In [1]:
# Cell 1: Install fine-tuning/runtime extras only. Do NOT reinstall numpy/pandas/sklearn on Kaggle.
import sys
import subprocess

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "-U",
    "transformers>=4.51.0",
    "accelerate>=0.34.0",
    "bitsandbytes",
    "peft>=0.13.0",
    "datasets>=2.20.0",
    "sentencepiece",
    "safetensors",
    "huggingface_hub",
])

print("Installed Qwen fine-tuning extras without touching Kaggle numpy/pandas/sklearn.")

try:
    import numpy as np
    import pandas as pd
    import sklearn
    print("NumPy:", np.__version__, "Pandas:", pd.__version__, "Sklearn:", sklearn.__version__)
except Exception as exc:
    raise RuntimeError("Kaggle scientific stack is ABI-broken. Restart/factory reset the session, then rerun. Original error: " + str(exc))


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 89.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 33.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 44.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 33.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 693.4/693.4 kB 36.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.6/116.6 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 105.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 97.4 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which

Installed Qwen fine-tuning extras without touching Kaggle numpy/pandas/sklearn.
NumPy: 2.4.6 Pandas: 2.3.3 Sklearn: 1.6.1


In [2]:
# Cell 2: Reconstruct the backend package from the embedded project payload
import base64
import io
import os
import pathlib
import shutil
import sys
import zipfile

BASE_DIR = pathlib.Path("/kaggle/working/tata_steel_agentic_ai")
if BASE_DIR.exists():
    shutil.rmtree(BASE_DIR)
BASE_DIR.mkdir(parents=True, exist_ok=True)

PROJECT_ZIP_B64_CHUNKS = [
  '...',
  'AA8AxQMAABHQAQAAAA=='
]

payload = base64.b64decode("".join(PROJECT_ZIP_B64_CHUNKS))
with zipfile.ZipFile(io.BytesIO(payload), "r") as zf:
    zf.extractall(BASE_DIR)

sys.path.insert(0, str(BASE_DIR))
print("Reconstructed backend at", BASE_DIR)
print("Backend files:")
for path in sorted((BASE_DIR / "backend").glob("*.py")):
    print("-", path.name)


Reconstructed backend at /kaggle/working/tata_steel_agentic_ai
Backend files:
- __init__.py
- agent.py
- api.py
- config.py
- data_setup.py
- decision_intelligence.py
- dynamic_assets.py
- finetune_data.py
- llm.py
- models.py
- rag.py
- steel_agent.py


In [3]:
# Cell 3: Runtime configuration
import os
import gc
import json
import torch
from pathlib import Path

MODEL_ID = "Qwen/Qwen3-8B"
# For a quick dry run on smaller GPU, temporarily switch to:
# MODEL_ID = "Qwen/Qwen3-4B"

OUTPUT_DIR = Path("/kaggle/working/qwen3_8b_steel_maintenance_lora")
DATASET_JSONL = Path("/kaggle/working/qwen3_8b_maintenance_sft.jsonl")
MAX_LENGTH = 2048
LORA_R = 8
LORA_ALPHA = 16
NUM_EPOCHS = 1
LEARNING_RATE = 5e-5
GRAD_ACCUM = 8

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU count:", torch.cuda.device_count())
    for i in range(torch.cuda.device_count()):
        free, total = torch.cuda.mem_get_info(i)
        print(f"GPU {i}: {torch.cuda.get_device_name(i)} | free {free/1024**3:.2f} GiB / total {total/1024**3:.2f} GiB")
else:
    raise RuntimeError("Enable GPU in Kaggle settings before fine-tuning Qwen3-8B.")


CUDA available: True
GPU count: 2
GPU 0: Tesla T4 | free 14.46 GiB / total 14.56 GiB
GPU 1: Tesla T4 | free 14.46 GiB / total 14.56 GiB


In [4]:
# Cell 4: Generate maintenance data, decision-intelligence table, and Qwen SFT dataset
import json
import pandas as pd

from backend.data_setup import prepare_data
from backend.decision_intelligence import build_decision_intelligence_table, top_original_vs_dynamic
from backend.finetune_data import build_qwen_sft_examples, load_sft_jsonl

summary = prepare_data(force=True)
print("Data summary:")
print(json.dumps(summary, indent=2))

decision_table = build_decision_intelligence_table(force=True)
print("Decision-intelligence rows:", len(decision_table))
display(decision_table[[
    "asset_id", "asset_type", "area", "active", "is_dynamic", "decision_score", "priority", "risk_band",
    "estimated_rul_days", "evidence_confidence", "procurement_risk", "applied_rule_count"
]].head(20))

examples = build_qwen_sft_examples(DATASET_JSONL, min_examples=80)
print("SFT examples:", len(examples))
print("Saved dataset:", DATASET_JSONL)
print("Sample example:")
print(json.dumps(examples[0], indent=2, ensure_ascii=False)[:3000])


Data summary:
{
  "steel_rows": 2880,
  "public_rows": 10000,
  "combined_rows": 12880,
  "public_available": true
}
Decision-intelligence rows: 4


,asset_id,asset_type,area,active,is_dynamic,decision_score,priority,risk_band,estimated_rul_days,evidence_confidence,procurement_risk,applied_rule_count
0,MTR-204,Induction Motor,Hot Strip Mill,True,0,68.99,P2,HIGH,14.1,HIGH,HIGH,0
1,GBX-17,Gearbox,Finishing Mill,True,0,68.55,P2,HIGH,14.7,HIGH,HIGH,0
2,PMP-09,Cooling Water Pump,Caster Utility,True,0,56.52,P3,MEDIUM,18.9,HIGH,HIGH,0
3,HPP-12,Hydraulic Power Pack,Plate Mill,True,0,39.56,P3,MEDIUM,23.8,HIGH,HIGH,0


SFT examples: 80
Saved dataset: /kaggle/working/qwen3_8b_maintenance_sft.jsonl
Sample example:
{
  "messages": [
    {
      "role": "system",
      "content": "You are a serious agentic AI maintenance copilot for steel manufacturing. Use only the provided evidence. Do not invent history, SOPs, spares, sensor values, or root causes. When evidence is missing, say what is missing and give a safe next action."
    },
    {
      "role": "user",
      "content": "Create a maintenance decision report for MTR-204. Use this evidence packet and do not invent missing data:\n{\"decision\": {\"asset_id\": \"MTR-204\", \"asset_type\": \"Induction Motor\", \"area\": \"Hot Strip Mill\", \"criticality\": \"High\", \"active\": true, \"is_dynamic\": 0, \"data_origin\": \"steel_demo_app\", \"temperature\": 91.988, \"vibration\": 5.693, \"current\": 80.858, \"pressure\": 9.056, \"rpm\": 1436.169, \"alarm_count\": 3, \"raw_health_score\": 59.74, \"rule_adjusted_score\": 59.74, \"decision_score\": 68.99, \

In [5]:
# Cell 5: Load Qwen3-8B in 4-bit and attach LoRA adapters
import gc
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from huggingface_hub import login

hf_token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_TOKEN")
if hf_token:
    login(token=hf_token, add_to_git_credential=False)

def supports_bf16():
    if not torch.cuda.is_available():
        return False
    major, _minor = torch.cuda.get_device_capability(0)
    return major >= 8

compute_dtype = torch.bfloat16 if supports_bf16() else torch.float16

gc.collect()
torch.cuda.empty_cache()

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True, use_fast=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

max_memory = {}
for i in range(torch.cuda.device_count()):
    free, total = torch.cuda.mem_get_info(i)
    max_memory[i] = f"{max(1, int((free / 1024**3) - 1.5))}GiB"
max_memory["cpu"] = "24GiB"
print("max_memory:", max_memory)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    device_map="auto",
    max_memory=max_memory,
    quantization_config=bnb_config,
    torch_dtype=compute_dtype,
    low_cpu_mem_usage=True,
    use_safetensors=True,
    attn_implementation="eager",
)
model.config.use_cache = False
model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/32.9k [00:00<?, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

trainable params: 21,823,488 || all params: 8,212,558,848 || trainable%: 0.2657


In [6]:
# Cell 6: Tokenize chat examples with assistant-only loss masking
from datasets import Dataset
from transformers import default_data_collator

records = load_sft_jsonl(DATASET_JSONL)
print("Loaded records:", len(records))

def render_messages(messages, include_assistant=True):
    if include_assistant:
        used = messages
        add_generation_prompt = False
    else:
        used = messages[:-1]
        add_generation_prompt = True
    try:
        return tokenizer.apply_chat_template(
            used,
            tokenize=False,
            add_generation_prompt=add_generation_prompt,
            enable_thinking=False,
        )
    except TypeError:
        return tokenizer.apply_chat_template(
            used,
            tokenize=False,
            add_generation_prompt=add_generation_prompt,
        )


def tokenize_record(record):
    messages = record["messages"]
    full_text = render_messages(messages, include_assistant=True)
    prompt_text = render_messages(messages, include_assistant=False)
    if tokenizer.eos_token and not full_text.endswith(tokenizer.eos_token):
        full_text += tokenizer.eos_token

    full = tokenizer(full_text, truncation=True, max_length=MAX_LENGTH, padding=False)
    prompt = tokenizer(prompt_text, truncation=True, max_length=MAX_LENGTH, padding=False)
    labels = full["input_ids"].copy()
    prompt_len = min(len(prompt["input_ids"]), len(labels))
    labels[:prompt_len] = [-100] * prompt_len
    full["labels"] = labels
    return full

raw_ds = Dataset.from_list(records)
tok_ds = raw_ds.map(tokenize_record, remove_columns=raw_ds.column_names)
split = tok_ds.train_test_split(test_size=0.08, seed=42)
print(split)
print("Example token length:", len(split["train"][0]["input_ids"]))


Loaded records: 80


Map:   0%|          | 0/80 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 73
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 7
    })
})
Example token length: 186


In [7]:
# Cell 7: Train LoRA adapter
from transformers import Trainer, TrainingArguments

args = TrainingArguments(
    output_dir=str(OUTPUT_DIR / "trainer_checkpoints"),
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    logging_steps=5,
    eval_strategy="steps",
    eval_steps=25,
    save_strategy="epoch",
    optim="paged_adamw_8bit",
    fp16=not supports_bf16(),
    bf16=supports_bf16(),
    gradient_checkpointing=True,
    report_to="none",
    remove_unused_columns=False,
    max_grad_norm=0.3,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=split["train"],
    eval_dataset=split["test"],
    data_collator=default_data_collator,
)

trainer.train()
print("Training complete")


[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Step,Training Loss,Validation Loss
10,2.403996,2.233819


Training complete


In [8]:
# Cell 8: Save adapter, tokenizer, dataset, and metadata
import json
import shutil
from pathlib import Path

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
shutil.copy(DATASET_JSONL, OUTPUT_DIR / DATASET_JSONL.name)

metadata = {
    "base_model": MODEL_ID,
    "method": "QLoRA / PEFT LoRA adapter",
    "dataset_jsonl": str(DATASET_JSONL),
    "num_examples": len(records),
    "max_length": MAX_LENGTH,
    "lora_r": LORA_R,
    "lora_alpha": LORA_ALPHA,
    "epochs": NUM_EPOCHS,
    "learning_rate": LEARNING_RATE,
    "intended_use": "Tata Steel Maintenance Wizard agent planning and final maintenance-answer synthesis",
    "warning": "Adapter is trained on synthetic steel demo data plus public benchmark-derived instruction examples; do not claim proprietary Tata Steel data.",
}
(OUTPUT_DIR / "adapter_metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
print("Saved adapter to:", OUTPUT_DIR)
print(json.dumps(metadata, indent=2))


Saved adapter to: /kaggle/working/qwen3_8b_steel_maintenance_lora
{
  "base_model": "Qwen/Qwen3-8B",
  "method": "QLoRA / PEFT LoRA adapter",
  "dataset_jsonl": "/kaggle/working/qwen3_8b_maintenance_sft.jsonl",
  "num_examples": 80,
  "max_length": 2048,
  "lora_r": 8,
  "lora_alpha": 16,
  "epochs": 1,
  "learning_rate": 5e-05,
  "intended_use": "Tata Steel Maintenance Wizard agent planning and final maintenance-answer synthesis",
  "warning": "Adapter is trained on synthetic steel demo data plus public benchmark-derived instruction examples; do not claim proprietary Tata Steel data."
}


In [9]:
# Cell 9: Grounded inference smoke test with the fine-tuned adapter loaded in memory
# Important: the fine-tuned model should not answer from memory alone. It should write from a locked evidence packet.
import json
import torch
import pandas as pd

from backend.decision_intelligence import build_decision_intelligence_table, top_original_vs_dynamic

# Rebuild/read the same factual packet the agent would give to Qwen.
decision_table = build_decision_intelligence_table(force=True)
comparison_packet = top_original_vs_dynamic()

# Keep the packet compact so Qwen sees the actual facts instead of guessing.
cols = [
    "asset_id", "asset_type", "area", "active", "is_dynamic", "decision_score", "priority", "risk_band",
    "failure_risk", "estimated_rul_days", "evidence_confidence", "missing_evidence",
    "procurement_risk", "spare_plan", "delay_cost_impact_inr", "applied_rule_count", "next_system_action",
]
ranked_packet = decision_table[cols].head(8).to_dict("records")

evidence_packet = {
    "task": "choose exactly one asset for immediate maintenance today",
    "ranking_scope": "original demo assets plus active dynamic assets",
    "top_original_vs_dynamic": comparison_packet,
    "ranked_assets": ranked_packet,
    "hard_constraints": [
        "Use only this evidence packet.",
        "Do not invent maintenance history, SOPs, spares, sensor values, or root causes.",
        "If evidence is REVIEW or missing, say so clearly.",
        "Choose exactly one final asset.",
        "Keep the answer concise and field-ready.",
    ],
}

prompt = """Using the evidence packet below, answer the maintenance engineer.

Question: If I can maintain only one asset today, which one should I choose and why? Include evidence confidence, RUL, procurement risk, and next action.

Evidence packet:
{packet}

Write the final answer in natural engineer-facing language. No JSON. No long numeric lists. No made-up facts.
""".format(packet=json.dumps(evidence_packet, indent=2, default=str, allow_nan=False))

messages = [
    {"role": "system", "content": "You are a serious agentic AI maintenance copilot for steel manufacturing. Use only provided evidence and be concise. Never output long numeric sequences."},
    {"role": "user", "content": prompt + "\n/no_think"},
]

try:
    rendered = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
except TypeError:
    rendered = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

inputs = tokenizer(
    rendered,
    return_tensors="pt",
    truncation=True,
    max_length=MAX_LENGTH,
).to(model.device)

model.eval()
with torch.no_grad():
    out = model.generate(
        **inputs,
        max_new_tokens=420,
        do_sample=False,
        repetition_penalty=1.18,
        no_repeat_ngram_size=5,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

answer = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
answer = answer.replace("</think>", "").strip()
print(answer)

# Basic sanity checks: catch repetition failure before trusting the adapter.
assert "1200/1201/1202" not in answer, "Generation collapsed into numeric repetition. Rerun with a lower LR/less overfit or use the base model plus adapter cautiously."
assert len(answer.split()) >= 25, "Answer is too short to validate."
print("\nSmoke test passed: grounded adapter generated a non-repetitive answer.")


(no markdown)."
  ]
}


Asset MTR-203 (Induction Motor in Hot Strip Mill) - High criticality, failure risk 59.8%, estimated RUL 14 days, HIGH procurement risk. Next action: Schedule inspection within 26h, verify spare readiness (DE bearing 6321), check cooling path & alignment. Delay cost impact INR 5.95M. Confidence: HIGH.

Smoke test passed: grounded adapter generated a non-repetitive answer.


In [10]:
# Cell 10: Zip adapter for download or Kaggle dataset creation
import shutil
zip_path = shutil.make_archive("/kaggle/working/qwen3_8b_steel_maintenance_lora", "zip", OUTPUT_DIR)
print("Saved zip:", zip_path)
print("Download this zip or create a Kaggle Dataset from the adapter directory if you want to reuse it later.")


Saved zip: /kaggle/working/qwen3_8b_steel_maintenance_lora.zip
Download this zip or create a Kaggle Dataset from the adapter directory if you want to reuse it later.
